[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/23_attention_solution.ipynb)

# 🟡 Solution: Scaled Dot-Product Attention

*Attention & Transformers · Medium*

Reference implementation. Try it yourself in `23_attention.ipynb` first.

---
Implement **scaled dot-product attention** as a plain function.

$$\text{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

### Signature
`scaled_dot_product_attention(q, k, v, mask=None)`

| tensor | shape | notes |
|---|---|---|
| `q` | `(B, T_q, d_k)` | queries |
| `k` | `(B, T_k, d_k)` | keys — same feature dim as `q` |
| `v` | `(B, T_k, d_v)` | values — `d_v` may differ from `d_k` |
| `mask` | broadcastable to `(B, T_q, T_k)` | boolean, `True` = **attend**, `False` = **block** |
| returns | `(B, T_q, d_v)` | |

### Rules
- No `jax.nn.dot_product_attention`, no `nnx.MultiHeadAttention`
- `jax.nn.softmax` is allowed and encouraged (it subtracts the row max for you)
- `T_q` and `T_k` are independent — do not assume a square score matrix
- Only the **last two** axes are contracted, so the identical code must also
  accept `(B, H, T, D_h)` inputs. Use `jnp.swapaxes(k, -1, -2)`, never a
  hard-coded `transpose(0, 2, 1)`
- Masked positions are removed **before** the softmax, not zeroed after it

### Why the $1/\sqrt{d_k}$ is not cosmetic
Take $q, k$ with i.i.d. zero-mean unit-variance entries. Their dot product is a
sum of $d_k$ independent unit-variance terms, so

$$\operatorname{Var}(q \cdot k) = d_k, \qquad \operatorname{std}(q \cdot k) = \sqrt{d_k}$$

At $d_k = 64$ the raw logits have standard deviation $8$, so the gap between the
largest and a typical logit is routinely $20$+. `softmax` of that is numerically
one-hot. And the Jacobian of softmax is $\operatorname{diag}(p) - pp^\top$,
which vanishes as $p$ becomes one-hot — so the layer stops passing gradient
before training has learned anything. Dividing by $\sqrt{d_k}$ pulls the logit
variance back to $1$ regardless of head width, which is exactly why you can
widen heads without retuning the initialisation.

This is the single most common thing interviewers check, along with whether you
scale by $\sqrt{d_k}$ (the **key** dim, per head) rather than $\sqrt{d_{model}}$.
In multi-head attention those differ by a factor of $H$.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.shape[-1]

    # swapaxes(-1, -2) transposes only the last two axes, so this works for
    # (B, T, D) and for (B, H, T, Dh) without changing a line.
    scores = (q @ jnp.swapaxes(k, -1, -2)) / jnp.sqrt(jnp.asarray(d_k, q.dtype))

    if mask is not None:
        # Blocked entries get a large negative logit BEFORE the softmax, so the
        # normaliser never sees them at all.
        scores = jnp.where(mask, scores, jnp.asarray(-1e9, scores.dtype))

    weights = jax.nn.softmax(scores, axis=-1)   # subtracts the row max for us
    return weights @ v

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

# What the scaling actually buys you: logit spread vs head width.
key = jax.random.key(0)
for d_k in (8, 64, 512):
    kq, kk = jax.random.split(jax.random.key(d_k))
    q = jax.random.normal(kq, (1, 1, d_k))
    k = jax.random.normal(kk, (1, 64, d_k))
    raw = (q @ jnp.swapaxes(k, -1, -2))[0, 0]
    scaled = raw / jnp.sqrt(float(d_k))
    p_raw = jax.nn.softmax(raw)
    p_scaled = jax.nn.softmax(scaled)
    print(f"d_k={d_k:4d}  logit std raw={raw.std():7.2f} scaled={scaled.std():5.2f}"
          f"  max prob raw={p_raw.max():.3f} scaled={p_scaled.max():.3f}")

# Cross shapes: 3 queries attending over 5 keys, values 8-dim.
q = jax.random.normal(jax.random.key(1), (2, 3, 16))
k = jax.random.normal(jax.random.key(2), (2, 5, 16))
v = jax.random.normal(jax.random.key(3), (2, 5, 8))
print("out:", scaled_dot_product_attention(q, k, v).shape)   # (2, 3, 8)

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("attention")